## Calling API from OpneRouter


In [7]:
!pip install openai requests -q

In [8]:
import os
import json
import re
import time
import requests
import subprocess
from openai import OpenAI
from google.colab import userdata

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)
FREE_MODEL = "openrouter/free"
TOOL_MODEL = "openrouter/free"

### Your first API call

This is **inference**. You send tokens, get tokens back.  the API just wraps it with HTTP.

In [9]:
response = client.chat.completions.create(
    model=FREE_MODEL,
    messages=[{"role": "user", "content": "Hii"}],
    temperature=0.0,
    max_tokens=50,
)

print("Response:", response.choices[0].message.content)
print(f"Tokens — in: {response.usage.prompt_tokens}, out: {response.usage.completion_tokens}")
print(f"Model used: {response.model}")

Response: None
Tokens — in: 16, out: 50
Model used: poolside/laguna-m.1-20260312:free


### The agent's system prompt

This prompt **IS** the architecture. It tells the model to alternate between THOUGHT and ACTION in a parseable format.

In [10]:
REACT_SYSTEM_PROMPT = """You are a helpful assistant that solves problems step by step using tools.

You have access to these tools:
{tool_descriptions}

## How to respond

When you need to use a tool, respond in EXACTLY this format:

THOUGHT: <your reasoning about what to do next>
ACTION: <tool_name>
ACTION_INPUT: <arguments as valid JSON>

When you have enough information for the final answer:

THOUGHT: <your final reasoning>
FINAL_ANSWER: <your complete answer to the user>

## Rules
- Always start with THOUGHT
- Use only ONE action per turn
- Wait for the OBSERVATION before continuing
- If a tool returns an error, reason about it and try a different approach
- Be concise in your thoughts
"""

### Real tools

Wikipedia search hits a real API. The calculator does real math. These are the "hands" of our agent.

In [20]:
from urllib.parse import quote
from datetime import datetime
import requests
import json

# Search Wikipedia
def search_wikipedia(query):
    """
    Search Wikipedia robustly. First tries direct summary, then general search if needed.
    """
    # Try direct summary API first
    summary_url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{quote(query)}"
    try:
        r = requests.get(summary_url, timeout=10)
        if r.status_code == 200:
            data = r.json()
            if data.get("extract"): # Ensure there's an extract
                return json.dumps({
                    "title": data.get("title", query),
                    "summary": data.get("extract", "No summary found.")[:800]
                })
    except requests.exceptions.RequestException:
        pass # Fallback to general search if direct summary fails or timeouts

    # If direct summary fails or is not found, try a general search
    search_url = (
        f"https://en.wikipedia.org/w/api.php?"
        f"action=query&format=json&list=search&srsearch={quote(query)}&srlimit=1" # Limit to 1 result
    )
    try:
        r = requests.get(search_url, timeout=10)
        if r.status_code == 200:
            search_data = r.json()
            search_results = search_data.get("query", {}).get("search", [])
            if search_results:
                best_title = search_results[0]["title"]
                # Now fetch summary for the best_title
                summary_url_from_search = f"https://en.wikipedia.org/api/rest_v1/page/summary/{quote(best_title)}"
                r_summary = requests.get(summary_url_from_search, timeout=10)
                if r_summary.status_code == 200:
                    data = r_summary.json()
                    return json.dumps({
                        "title": data.get("title", best_title),
                        "summary": data.get("extract", "No summary found.")[:800]
                    })
        return json.dumps({
            "error": f"No definitive Wikipedia page found for '{query}'."
        })
    except Exception as e:
        return json.dumps({
            "error": f"Error during robust Wikipedia search: {str(e)}"
        })


# Calculator
def calculate_math(expression):
    """Safely evaluate a mathematical expression."""

    try:
        allowed = set("0123456789+-*/.() eE")

        if not all(c in allowed for c in expression):
            return json.dumps({
                "error": f"Invalid characters in '{expression}'."
            })

        result = eval(expression)

        return json.dumps({
            "expression": expression,
            "result": round(result, 6)
        })

    except Exception as e:
        return json.dumps({
            "error": str(e)
        })


# Current Date & Time
def get_current_date():
    """Return the current date and time."""

    return json.dumps({
        "datetime": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    })


# Tool Registry
TOOLS = {
    "search_wikipedia": {
        "fn": search_wikipedia,
        "desc": "Search Wikipedia and return a short summary."
    },

    "calculate": {
        "fn": calculate_math,
        "desc": "Evaluate a mathematical expression."
    },

    "get_current_date": {
        "fn": get_current_date,
        "desc": "Return the current date and time."
    },
}

print("Registered Tools:", list(TOOLS.keys()))

Registered Tools: ['search_wikipedia', 'calculate', 'get_current_date']


In [12]:
def run_agent(user_query, max_iterations=10, verbose=True):
    """
    A ReAct Agent built from scratch.
    Uses an LLM, external tools, and an agent loop.
    """

    # Build the system prompt with available tool descriptions
    tool_desc = "\n".join(f"- {tool['desc']}" for tool in TOOLS.values())
    system = REACT_SYSTEM_PROMPT.format(tool_descriptions=tool_desc)

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_query},
    ]

    if verbose:
        print("\n" + "=" * 70)
        print(f"🧑 USER: {user_query}")
        print("=" * 70)

    for i in range(max_iterations):

        if verbose:
            print(f"\n🔄 Iteration {i + 1}/{max_iterations}")

        # STEP 1: Ask the LLM what to do next
        try:
            response = client.chat.completions.create(
                model=FREE_MODEL,
                messages=messages,
                temperature=0,
                max_tokens=800,
            )

        except Exception as e:
            print(f"\n❌ API Error: {e}")
            return {
                "answer": "Failed to communicate with the language model.",
                "iterations": i + 1,
            }

        text = (response.choices[0].message.content or "").strip()

        messages.append({
            "role": "assistant",
            "content": text
        })

        if verbose:
            print("\n🤖 MODEL RESPONSE")
            print("-" * 70)
            print(text)
            print("-" * 70)

        # STEP 2: Extract the THOUGHT
        thought_match = re.search(
            r"THOUGHT:\s*(.+?)(?=ACTION:|FINAL_ANSWER:|$)",
            text,
            re.DOTALL,
        )

        thought = thought_match.group(1).strip() if thought_match else ""

        if verbose and thought:
            print(f"\n💭 THOUGHT: {thought}")

        # STEP 3: Check if the model has finished
        if "FINAL_ANSWER:" in text:

            answer = text.split("FINAL_ANSWER:")[-1].strip()

            if verbose:
                print("\n" + "=" * 70)
                print(f"✅ AGENT FINISHED IN {i + 1} ITERATION(S)")
                print("=" * 70)
                print(answer)

            return {
                "answer": answer,
                "iterations": i + 1,
            }

        # STEP 4: Extract ACTION and ACTION_INPUT
        action_match = re.search(r"ACTION:\s*(\w+)", text)
        input_match = re.search(
            r"ACTION_INPUT:\s*(.+?)(?:\n|$)",
            text,
            re.DOTALL,
        )

        if not action_match:

            if verbose:
                print("\n⚠️ Invalid format returned by the model. Asking again...")

            messages.append({
                "role": "user",
                "content": (
                    "Please respond using either:\n"
                    "ACTION + ACTION_INPUT\n"
                    "or\n"
                    "FINAL_ANSWER."
                ),
            })

            continue

        tool_name = action_match.group(1).strip()
        raw_input = input_match.group(1).strip() if input_match else "{}"

        # STEP 5: Execute the selected tool
        if tool_name not in TOOLS:

            observation = json.dumps({
                "error": f"Unknown tool '{tool_name}'. Available tools: {list(TOOLS.keys())}"
            })

        else:

            try:

                if raw_input.startswith("{"):
                    args = json.loads(raw_input)
                else:
                    args = {"query": raw_input.strip("\"'")}

                observation = TOOLS[tool_name]["fn"](**args)

            except Exception as e:

                observation = json.dumps({
                    "error": f"Failed to execute '{tool_name}': {str(e)}"
                })

        if verbose:
            print(f"\n🔧 ACTION: {tool_name}")
            print(f"📥 INPUT : {raw_input}")

            obs_preview = (
                observation[:300] + "..."
                if len(observation) > 300
                else observation
            )

            print(f"👁️ OBSERVATION: {obs_preview}")

        # STEP 6: Send the observation back to the model
        messages.append({
            "role": "user",
            "content": f"OBSERVATION: {observation}"
        })

    if verbose:
        print(f"\n⚠️ Maximum iterations ({max_iterations}) reached.")

    return {
        "answer": "Maximum iterations reached.",
        "iterations": max_iterations,
    }

##Trace & Analyze the Agent

Let's instrument the agent to understand **costs, token growth, and behavior patterns**. This is the context engineering problem — every iteration re-sends everything.

In [13]:
def run_agent_traced(user_query, max_iterations=10):
    """Same ReAct agent, but collects detailed execution traces."""

    tool_desc = "\n".join(f"- {t['desc']}" for t in TOOLS.values())
    system = REACT_SYSTEM_PROMPT.format(tool_descriptions=tool_desc)

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_query},
    ]

    trace = []
    token_log = []

    for i in range(max_iterations):

        try:
            response = client.chat.completions.create(
                model=FREE_MODEL,
                messages=messages,
                temperature=0,
                max_tokens=800,
            )

        except Exception as e:
            trace.append({
                "step": i + 1,
                "type": "❌ API ERROR",
                "error": str(e)
            })
            break

        # Token Usage
        usage = response.usage

        token_log.append({
            "iter": i + 1,
            "in": usage.prompt_tokens,
            "out": usage.completion_tokens,
        })

        text = (response.choices[0].message.content or "").strip()

        messages.append({
            "role": "assistant",
            "content": text
        })

        # Extract Thought
        thought_match = re.search(
            r"THOUGHT:\s*(.+?)(?=ACTION:|FINAL_ANSWER:|$)",
            text,
            re.DOTALL,
        )

        thought = thought_match.group(1).strip() if thought_match else ""

        # Final Answer
        if "FINAL_ANSWER:" in text:

            answer = text.split("FINAL_ANSWER:")[-1].strip()

            trace.append({
                "step": i + 1,
                "type": "✅ FINAL",
                "thought": thought,
                "answer": answer,
            })

            break

        # Extract Action
        action_match = re.search(r"ACTION:\s*(\w+)", text)

        input_match = re.search(
            r"ACTION_INPUT:\s*(.+?)(?:\n|$)",
            text,
            re.DOTALL,
        )

        if not action_match:

            messages.append({
                "role": "user",
                "content": "Please respond using ACTION + ACTION_INPUT or FINAL_ANSWER."
            })

            trace.append({
                "step": i + 1,
                "type": "⚠️ FORMAT",
                "thought": thought,
            })

            continue

        tool_name = action_match.group(1).strip()
        raw_input = input_match.group(1).strip() if input_match else "{}"

        # Execute Tool
        if tool_name in TOOLS:

            try:

                if raw_input.startswith("{"):
                    args = json.loads(raw_input)
                else:
                    args = {"query": raw_input.strip("\"'")}

                observation = TOOLS[tool_name]["fn"](**args)

            except Exception as e:

                observation = json.dumps({
                    "error": str(e)
                })

        else:

            observation = json.dumps({
                "error": f"Unknown tool: {tool_name}"
            })

        trace.append({
            "step": i + 1,
            "type": "🔧 ACTION",
            "thought": thought[:150],
            "tool": tool_name,
            "input": raw_input[:100],
            "observation": observation[:200],
        })

        messages.append({
            "role": "user",
            "content": f"OBSERVATION: {observation}"
        })

    return trace, token_log

In [14]:
def read_file(path):
    """Read a file's contents."""
    try:
        with open(path, "r") as f:
            content = f.read()
        return json.dumps({"path": path, "content": content[:3000], "truncated": len(content) > 3000})
    except Exception as e:
        return json.dumps({"error": str(e)})

def write_file(path, content):
    """Write content to a file."""
    try:
        d = os.path.dirname(path)
        if d:
            os.makedirs(d, exist_ok=True)
        with open(path, "w") as f:
            f.write(content)
        return json.dumps({"status": "success", "path": path, "bytes": len(content)})
    except Exception as e:
        return json.dumps({"error": str(e)})

def run_python(code):
    """Execute Python code and return stdout/stderr."""
    try:
        result = subprocess.run(
            ["python3", "-c", code],
            capture_output=True, text=True, timeout=15,
        )
        return json.dumps({
            "stdout": result.stdout[:2000] if result.stdout else "",
            "stderr": result.stderr[:2000] if result.stderr else "",
            "exit_code": result.returncode,
        })
    except subprocess.TimeoutExpired:
        return json.dumps({"error": "Timed out (15s limit)"})
    except Exception as e:
        return json.dumps({"error": str(e)})

def list_files(path="."):
    """List files in a directory."""
    try:
        items = sorted(os.listdir(path))
        return json.dumps({"path": path, "files": items[:50]})
    except Exception as e:
        return json.dumps({"error": str(e)})

CODE_TOOLS = {
    "read_file": {"fn": read_file, "desc": "read_file(path: str) — Read a file and return its contents."},
    "write_file": {"fn": write_file, "desc": 'write_file(path: str, content: str) — Write content to a file.'},
    "run_python": {"fn": run_python, "desc": "run_python(code: str) — Execute Python code. Returns stdout/stderr."},
    "list_files": {"fn": list_files, "desc": "list_files(path: str) — List files in a directory."},
    "calculate": {"fn": calculate_math, "desc": "calculate(expression: str) — Evaluate a math expression."},
}

print(f"Code tools: {list(CODE_TOOLS.keys())}")

Code tools: ['read_file', 'write_file', 'run_python', 'list_files', 'calculate']


In [15]:
CODE_SYSTEM_PROMPT = """You are a coding agent. You write, run, and debug Python code to solve tasks.

Available tools:
{tool_descriptions}

Format — to use a tool:

THOUGHT: <your reasoning>
ACTION: <tool_name>
ACTION_INPUT: {{"arg1": "value1", "arg2": "value2"}}

Format — when done:

THOUGHT: <final reasoning>
FINAL_ANSWER: <your answer>

## Rules
- ONE action per turn. Wait for OBSERVATION.
- After writing code to a file, always run_python to TEST it.
- If a test fails, read the error, fix the code, try again.
- Verify your work before giving FINAL_ANSWER.
- Include print() statements so you can see output.
"""

def run_code_agent(user_query, max_iterations=15, verbose=True):
    """ReAct agent with coding tools."""
    tool_desc = "\n".join(f"- {t['desc']}" for t in CODE_TOOLS.values())
    system = CODE_SYSTEM_PROMPT.format(tool_descriptions=tool_desc)

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_query},
    ]

    if verbose:
        print(f"\n{'='*60}")
        print(f"🧑 TASK: {user_query}")
        print(f"{'='*60}")

    for i in range(max_iterations):
        if verbose:
            print(f"\n--- Iteration {i+1}/{max_iterations} ---")

        response = client.chat.completions.create(
            model=FREE_MODEL, messages=messages,
            temperature=0, max_tokens=1500,
        )

        text = response.choices[0].message.content or ""
        messages.append({"role": "assistant", "content": text})

        thought_match = re.search(r"THOUGHT:\s*(.+?)(?=ACTION:|FINAL_ANSWER:|$)", text, re.DOTALL)
        thought = thought_match.group(1).strip() if thought_match else ""
        if verbose and thought:
            print(f"💭 THOUGHT: {thought[:200]}")

        if "FINAL_ANSWER:" in text:
            answer = text.split("FINAL_ANSWER:")[-1].strip()
            if verbose:
                print(f"\n{'='*60}")
                print(f"✅ DONE in {i+1} iteration(s)")
                print(f"{'='*60}")
                print(f"📝 {answer[:500]}")
            return answer

        action_match = re.search(r"ACTION:\s*(\w+)", text)
        input_match = re.search(r"ACTION_INPUT:\s*(.+?)(?:\nTHOUGHT|\nACTION|\nFINAL|$)", text, re.DOTALL)

        if not action_match:
            messages.append({"role": "user", "content": "Use ACTION + ACTION_INPUT or FINAL_ANSWER."})
            if verbose:
                print("⚠️  Format issue, nudging...")
            continue

        tool_name = action_match.group(1).strip()
        raw_input = input_match.group(1).strip() if input_match else "{}"

        if verbose:
            print(f"🔧 ACTION: {tool_name}")

        if tool_name not in CODE_TOOLS:
            observation = json.dumps({"error": f"Unknown tool '{tool_name}'. Available: {list(CODE_TOOLS.keys())}"})
        else:
            try:
                args = json.loads(raw_input)
                observation = CODE_TOOLS[tool_name]["fn"](**args)
            except json.JSONDecodeError:
                try:
                    observation = CODE_TOOLS[tool_name]["fn"](raw_input.strip("\"'"))
                except Exception as e:
                    observation = json.dumps({"error": f"Parse + fallback failed: {e}"})
            except Exception as e:
                observation = json.dumps({"error": str(e)})

        if verbose:
            obs_short = observation[:250] + "..." if len(observation) > 250 else observation
            print(f"👁️  OBSERVATION: {obs_short}")

        messages.append({"role": "user", "content": f"OBSERVATION: {observation}"})

    return "Max iterations reached."

## Native Function-Calling Agent

Our Part 3 agent uses **string parsing** (regex on THOUGHT/ACTION). Modern APIs support **native function calling** — the model outputs structured `tool_calls` that don't need regex.

Let's rebuild using OpenRouter's tool calling API. This is closer to how production agents work.

In [16]:
NATIVE_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_wikipedia",
            "description": "Search Wikipedia for information about a topic.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Topic to search"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression like '(5+3)*2'"}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_date",
            "description": "Get the current date and time.",
            "parameters": {"type": "object", "properties": {}}
        }
    },
]

NATIVE_FNS = {
    "search_wikipedia": search_wikipedia,
    "calculate": calculate_math,
    "get_current_date": get_current_date,
}


def run_native_agent(user_query, max_iterations=10, verbose=True):
    """
    Agent using native function calling.
    No regex — the API returns structured tool_calls.
    """
    messages = [
        {"role": "system", "content":
            "You are a helpful research assistant. Use tools to find information. "
            "Think step by step. After gathering enough info, give a complete answer."},
        {"role": "user", "content": user_query},
    ]

    if verbose:
        print(f"\n{'='*60}")
        print(f"🧑 USER: {user_query}")
        print(f"{'='*60}")

    for i in range(max_iterations):
        if verbose:
            print(f"\n--- Iteration {i+1} ---")

        response = client.chat.completions.create(
            model=TOOL_MODEL, messages=messages,
            tools=NATIVE_TOOLS, temperature=0, max_tokens=800,
        )

        msg = response.choices[0].message
        finish = response.choices[0].finish_reason

        # Text response = done
        if finish == "stop" and msg.content:
            if verbose:
                print(f"✅ FINAL ANSWER:\n  {msg.content[:500]}")
            return {"answer": msg.content, "iterations": i + 1}

        # Tool calls
        if msg.tool_calls:
            messages.append(msg)

            for tc in msg.tool_calls:
                fn_name = tc.function.name
                fn_args = json.loads(tc.function.arguments) if tc.function.arguments else {}

                if verbose:
                    print(f"🔧 TOOL: {fn_name}({json.dumps(fn_args)[:80]})")

                if fn_name in NATIVE_FNS:
                    result = NATIVE_FNS[fn_name](**fn_args)
                else:
                    result = json.dumps({"error": f"Unknown tool: {fn_name}"})

                if verbose:
                    print(f"👁️  RESULT: {result[:150]}")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result,
                })
        else:
            if msg.content:
                messages.append({"role": "assistant", "content": msg.content})
            else:
                break

    return {"answer": "Max iterations reached", "iterations": max_iterations}

In [17]:
result = run_native_agent(
    "What's the date today and what will be tomorrow?"
)


🧑 USER: What's the date today and what will be tomorrow?

--- Iteration 1 ---
🔧 TOOL: get_current_date({})
👁️  RESULT: {"datetime": "2026-07-01 05:24:31"}

--- Iteration 2 ---
✅ FINAL ANSWER:
  Today's date is **2026-07-01** and tomorrow's date will be **2026-07-02**. Let me know if you need further assistance!


In [21]:
result_complex = run_native_agent(
    "What is the capital of France and what is its population? Calculate the population density if France's area is 643801 square kilometers."
)


🧑 USER: What is the capital of France and what is its population? Calculate the population density if France's area is 643801 square kilometers.

--- Iteration 1 ---
🔧 TOOL: search_wikipedia({"query": "France"})
👁️  RESULT: {"error": "No Wikipedia page found for 'France'."}

--- Iteration 2 ---
🔧 TOOL: search_wikipedia({"query": "France (country)"})
👁️  RESULT: {"error": "No Wikipedia page found for 'France (country)'."}

--- Iteration 3 ---
🔧 TOOL: search_wikipedia({"query": "Paris"})
👁️  RESULT: {"error": "No Wikipedia page found for 'Paris'."}

--- Iteration 4 ---
🔧 TOOL: search_wikipedia({"query": "France capital"})
👁️  RESULT: {"error": "No Wikipedia page found for 'France capital'."}

--- Iteration 5 ---
🔧 TOOL: calculate({"expression": "67842582 / 643801"})
👁️  RESULT: {"expression": "67842582 / 643801", "result": 105.378187}

--- Iteration 6 ---
✅ FINAL ANSWER:
  **Capital of France**  
- **Paris**

**Population of France**  
- Approximately **67.8 million** residents (2023‑20

### The 'Thought' Process in Native Function-Calling Agents

In the `run_agent` (ReAct) implementation, the model explicitly outputs a `THOUGHT:` field, which we then parse to understand its reasoning. However, in `run_native_agent`, which uses native function calling (like OpenRouter's tool calling API), the 'thought' process is more **internal** to the model.

The model still performs step-by-step reasoning, as guided by the system prompt (e.g., "Think step by step. After gathering enough info, give a complete answer."). This internal reasoning leads it directly to:

1.  **Decide to call a tool:** When the model determines it needs external information to answer the query.
2.  **Select the correct tool:** Based on its understanding of the available tools and the user's request.
3.  **Formulate tool arguments:** Generating the necessary parameters for the chosen tool.
4.  **Formulate a final answer:** Once it has sufficient information from tool observations.

So, while there isn't an explicit `THOUGHT:` output to parse, the model's 'thought' or 'meta' reasoning is embedded in its ability to choose and execute tools effectively to reach a solution. The `verbose=True` setting in `run_native_agent` allows you to observe this process indirectly by seeing the sequence of tool calls and their results, which reflects the model's internal progression towards the answer.

### Comparison: Simple Agent vs. Native Agent

| Feature | Simple Agent (ReAct) | Native Agent (Function Calling) |
| :--- | :--- | :--- |
| **Mechanism** | String Parsing (Regex) | Structured API Objects (JSON) |
| **Model Output** | `THOUGHT:`, `ACTION:`, `ACTION_INPUT:` | `tool_calls` array |
| **Robustness** | Fragile (Prone to typos/formatting errors) | High (Standardized JSON structure) |
| **Implementation** | `run_agent(user_query)` | `run_native_agent(user_query)` |
| **Communication** | Textual turn-taking | Native API tool integration |

**Summary:**
The **Simple Agent** acts like a reader who interprets a specific script, while the **Native Agent** acts like a system that communicates through standardized protocols. Native is preferred for production as it avoids the 'hallucinated' formatting errors common in text-based parsing.